# NYC Taxi Advanced Analytics

**Dataset:** `samples.nyctaxi.trips`

**Difficulty:** Hard

**Topics:** window functions, statistical analysis, time series, percentiles, sessionization

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

trips = spark.table("samples.nyctaxi.trips")

## Problem 1

**Rush Hour Fare Premium.**

Classify each trip as `rush_hour` (pickup between 7–9 AM or 5–7 PM on weekdays, i.e. day of week 2–6) or `off_peak` (everything else). Compare average fare, average distance, trip count, and total revenue for each category.

Business context: The taxi commission wants to understand whether rush-hour pricing generates a meaningful fare premium over off-peak periods to justify surge pricing policy.

**Expected output columns:** `time_category`, `trip_count`, `avg_fare`, `avg_distance`, `total_revenue`

In [ ]:
# Problem 1 — write your solution here
# Assign your result to: result_1

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None — did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'time_category' in cols, "Missing column: time_category"
assert 'trip_count' in cols, "Missing column: trip_count"
assert 'avg_fare' in cols, "Missing column: avg_fare"
assert 'total_revenue' in cols, "Missing column: total_revenue"
cnt = result_1.count()
assert cnt == 2, f"Expected exactly 2 rows (rush_hour + off_peak), got {cnt}"
categories = [r['time_category'] for r in result_1.select('time_category').collect()]
assert 'rush_hour' in categories and 'off_peak' in categories, "Expected both 'rush_hour' and 'off_peak' categories"
avg_fares = result_1.select('avg_fare').collect()
assert all(row['avg_fare'] > 0 for row in avg_fares), "avg_fare must be positive"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

**Trip Duration Anomaly Detection.**

Compute trip duration in minutes as the difference between `tpep_dropoff_datetime` and `tpep_pickup_datetime`. Then calculate the global mean and standard deviation of trip duration. Flag trips where the duration is more than 2 standard deviations from the mean as anomalies (`is_anomaly = true`).

Business context: Anomalous trips may indicate meter fraud, GPS errors, or data entry mistakes. Flagging them helps the quality-assurance team triage records before billing disputes.

**Expected output columns:** `tpep_pickup_datetime`, `trip_distance`, `fare_amount`, `duration_minutes`, `is_anomaly`

In [ ]:
# Problem 2 — write your solution here
# Assign your result to: result_2
# to compare each row against global stats

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None — did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'duration_minutes' in cols, "Missing column: duration_minutes"
assert 'is_anomaly' in cols, "Missing column: is_anomaly"
assert 'tpep_pickup_datetime' in cols, "Missing column: tpep_pickup_datetime"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
anomaly_count = result_2.filter(F.col('is_anomaly') == True).count()
assert anomaly_count > 0, "Expected at least some anomaly rows"
assert anomaly_count < cnt, "Not all rows should be anomalies"
print(f"Problem 2 passed ✓  ({cnt} rows, {anomaly_count} anomalies)")

## Problem 3

**Fare Efficiency Score.**

Compute for each `pickup_zip`: the average fare per mile (`avg_fare_per_mile = avg(fare_amount / trip_distance)`), the trip count, and rank each zip by `avg_fare_per_mile` descending (rank 1 = highest fare per mile). Filter out trips with zero distance.

Business context: Understanding which pickup zones generate the most revenue per mile helps dispatch optimisation — drivers should be nudged toward high-efficiency zones during off-peak hours.

**Expected output columns:** `pickup_zip`, `avg_fare_per_mile`, `trip_count`, `efficiency_rank`

In [ ]:
# Problem 3 — write your solution here
# Assign your result to: result_3
# finally use F.rank() over a Window ordered by avg_fare_per_mile desc

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None — did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'avg_fare_per_mile' in cols, "Missing column: avg_fare_per_mile"
assert 'efficiency_rank' in cols, "Missing column: efficiency_rank"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rank = result_3.agg(F.min('efficiency_rank')).collect()[0][0]
assert min_rank == 1, f"Minimum rank should be 1, got {min_rank}"
avg_fpm = result_3.agg(F.min('avg_fare_per_mile')).collect()[0][0]
assert avg_fpm > 0, "avg_fare_per_mile must be positive"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

**Busiest Pickup-Dropoff Corridors.**

Find the top 20 pickup–dropoff zip-code pairs. For each pair compute: trip count, average fare, average distance, and percentage of total trips (`pct_of_total_trips`). Sort by trip count descending.

Business context: Identifying the most-travelled corridors drives infrastructure investment decisions and can inform partnership deals with hotels or venues near these zip codes.

**Expected output columns:** `pickup_zip`, `dropoff_zip`, `trip_count`, `avg_fare`, `avg_distance`, `pct_of_total_trips`

In [ ]:
# Problem 4 — write your solution here
# Assign your result to: result_4
# a separate agg or Window, then compute pct, finally limit to top 20

result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None — did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'dropoff_zip' in cols, "Missing column: dropoff_zip"
assert 'pct_of_total_trips' in cols, "Missing column: pct_of_total_trips"
cnt = result_4.count()
assert cnt == 20, f"Expected exactly 20 rows (top 20 corridors), got {cnt}"
pct_sum = result_4.agg(F.sum('pct_of_total_trips')).collect()[0][0]
assert pct_sum <= 100.0, f"Sum of pct should be <= 100, got {pct_sum}"
assert pct_sum > 0, "pct_of_total_trips must be positive"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

**Hourly Trip Patterns by Day of Week.**

Compute trip count per `day_of_week` (use `F.dayofweek`: 1=Sunday … 7=Saturday in Spark) and `hour_of_day` (0–23). Use a window function to rank the hours within each day of week by trip count descending (`rank_within_day = 1` means the busiest hour for that day).

Business context: Demand forecasting requires knowing exactly which hours on which days are busiest, so that fleet managers can pre-position drivers and minimise passenger wait time.

**Expected output columns:** `day_of_week`, `hour_of_day`, `trip_count`, `rank_within_day`

In [ ]:
# Problem 5 — write your solution here
# Assign your result to: result_5
# then Window.partitionBy('day_of_week').orderBy(F.desc('trip_count')) + F.rank()

result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None — did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'day_of_week' in cols, "Missing column: day_of_week"
assert 'hour_of_day' in cols, "Missing column: hour_of_day"
assert 'trip_count' in cols, "Missing column: trip_count"
assert 'rank_within_day' in cols, "Missing column: rank_within_day"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rank = result_5.agg(F.min('rank_within_day')).collect()[0][0]
assert min_rank == 1, f"Minimum rank_within_day should be 1, got {min_rank}"
hour_range = result_5.agg(F.min('hour_of_day'), F.max('hour_of_day')).collect()[0]
assert hour_range[0] >= 0 and hour_range[1] <= 23, "hour_of_day must be in range 0-23"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

**Revenue Percentiles.**

For the top 10 busiest pickup zips (by trip count), compute the 25th, 50th, 75th, and 95th percentile of `fare_amount` using `F.percentile_approx`. Also include the total trip count per zip.

Business context: Understanding the fare distribution per zip-code helps pricing analysts set dynamic pricing floors and ceilings that are appropriate for each area's demand profile.


**Expected output columns:** `pickup_zip`, `p25`, `p50`, `p75`, `p95`, `trip_count`

In [ ]:
# Problem 6 — write your solution here
# Assign your result to: result_6
# F.percentile_approx('fare_amount', [0.25, 0.5, 0.75, 0.95]).alias('pcts')
# then use pcts[0], pcts[1], pcts[2], pcts[3] to extract individual percentiles

result_6 = None  # replace this

In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None — did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
for col in ['pickup_zip', 'p25', 'p50', 'p75', 'p95', 'trip_count']:
    assert col in cols, f"Missing column: {col}"
cnt = result_6.count()
assert cnt == 10, f"Expected exactly 10 rows (top 10 zips), got {cnt}"
row = result_6.orderBy('p50', ascending=False).first()
assert row['p25'] <= row['p50'] <= row['p75'] <= row['p95'], "Percentiles should be non-decreasing: p25 <= p50 <= p75 <= p95"
assert result_6.agg(F.min('trip_count')).collect()[0][0] > 0, "trip_count must be positive"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

**Distance Bucket Analysis.**

Bin trips into 5 distance buckets based on `trip_distance`: `<1mi`, `1-3mi`, `3-5mi`, `5-10mi`, `>10mi`. For each bucket compute: trip count, average fare, average duration in minutes (dropoff − pickup), and total revenue.

Business context: Pricing strategy varies by trip length — short trips have high fixed costs (e.g. city tolls) while long trips compete with car services. This analysis informs product-level pricing decisions.

**Expected output columns:** `distance_bucket`, `trip_count`, `avg_fare`, `avg_duration_mins`, `total_revenue`

In [ ]:
# Problem 7 — write your solution here
# Assign your result to: result_7
# compute duration_mins from timestamp difference, then groupBy + agg

result_7 = None  # replace this

In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None — did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'distance_bucket' in cols, "Missing column: distance_bucket"
assert 'trip_count' in cols, "Missing column: trip_count"
assert 'avg_duration_mins' in cols, "Missing column: avg_duration_mins"
assert 'total_revenue' in cols, "Missing column: total_revenue"
cnt = result_7.count()
assert cnt <= 5, f"Expected at most 5 distance buckets, got {cnt}"
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_7.agg(F.min('trip_count')).collect()[0][0] > 0, "trip_count must be positive"
assert result_7.agg(F.min('total_revenue')).collect()[0][0] > 0, "total_revenue must be positive"
print(f"Problem 7 passed ✓  ({cnt} rows)")

## Problem 8

**Rolling 7-Day Revenue per Pickup Zip.**

Compute daily total revenue (fare_amount) per pickup_zip, then calculate a 7-day rolling average of that daily revenue using a window function. Truncate `tpep_pickup_datetime` to day for the date column.

Business context: Smoothed revenue trends help operations teams identify genuine growth or decay in demand at a zip level, filtering out day-to-day noise such as weather events.


**Expected output columns:** `pickup_date`, `pickup_zip`, `daily_revenue`, `rolling_7day_avg`

In [ ]:
# Problem 8 — write your solution here
# Assign your result to: result_8
#   1. Truncate datetime to day, groupBy (pickup_zip, pickup_date) to get daily_revenue
#   2. Define a range-based window: Window.partitionBy('pickup_zip')
#        .orderBy(F.col('pickup_date').cast('long'))
#        .rangeBetween(-6*86400, 0)
#   3. Apply F.avg('daily_revenue').over(window) as rolling_7day_avg

result_8 = None  # replace this

In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────
assert result_8 is not None, "result_8 is None — did you assign your DataFrame?"
assert hasattr(result_8, 'columns'), "result_8 must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
assert 'pickup_date' in cols, "Missing column: pickup_date"
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'daily_revenue' in cols, "Missing column: daily_revenue"
assert 'rolling_7day_avg' in cols, "Missing column: rolling_7day_avg"
cnt = result_8.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_8.agg(F.min('daily_revenue')).collect()[0][0] > 0, "daily_revenue must be positive"
assert result_8.agg(F.min('rolling_7day_avg')).collect()[0][0] > 0, "rolling_7day_avg must be positive"
zip_count = result_8.select('pickup_zip').distinct().count()
assert zip_count > 1, f"Expected multiple pickup zips, got {zip_count}"
print(f"Problem 8 passed ✓  ({cnt} rows)")